# Day 9: Portion Size Estimation
Combining OpenCV Reference Object Scaling and a deep learning regression head to estimate food portion sizes.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.portion_estimator import ReferenceObjectPortionEstimator, RegressionPortionEstimator, scale_nutrition

## 1. Computer Vision (OpenCV) Reference Scaling Workflow

In [ ]:
os.makedirs('../data/sample', exist_ok=True)
sample_image_path = '../data/sample/mock_food.jpg'
# Create a dummy blank image for testing OpenCV block
cv2.imwrite(sample_image_path, np.ones((300, 300, 3), dtype=np.uint8) * 200)

cv_estimator = ReferenceObjectPortionEstimator(reference_real_width_cm=2.426) # Width of a US Quarter
estimated_grams_cv = cv_estimator.estimate_grams(sample_image_path, density_g_per_cm3=1.1)

print(f"OpenCV Estimated Portion: {estimated_grams_cv} grams")

## 2. Deep Learning Regression via EfficientNet

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dl_estimator = RegressionPortionEstimator().to(device)
dl_estimator.eval()

# Mock inference tensor representing an image
dummy_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    estimated_grams_dl = dl_estimator(dummy_input).item()
    
print(f"Deep Learning Initialized Regression Portion: {abs(estimated_grams_dl):.2f} grams")

## 3. Scaling USDA Nutrition Facts

In [ ]:
base_nutrition_100g = {
    "calories": 250.0,
    "protein_g": 10.0,
    "carbs_g": 30.0,
    "fat_g": 10.0
}

# Using the CV estimate for scaling
scaled_nutrients = scale_nutrition(base_nutrition_100g, estimated_grams_cv)
print("Base facts per 100g:", base_nutrition_100g)
print(f"Scaled facts for {estimated_grams_cv}g:", scaled_nutrients)